# Image Re-sizing for GAN fine tuning

**NOTE**: This notebook consists of two parts.  Image Selection for the GAN fine tuning can be run locally in order to resize for GAN consumption, but the images will need to be uploaded into Google Collab for the second half.  Using StyleGAN to generate images requires GPUs, and it is recommended to run in Google Collab to speed up should be the fine-tuning process.



In [1]:
# Import libraries used in the image re-sizing portion of the notebook
import os
import cv2
import numpy as np
import pandas as pd
import random
from matplotlib import pyplot as plt
from PIL import Image
from pathlib import Path


In [2]:
# Set File path for Image Data
data_filepath = './data/chest_xray'

In [3]:
# Create a Dataframe of image paths and their corresponding Label (Normal, Pnemonia)

# Use the TRAIN and TEST directories to build Dataframe
image_dirs = ['train', 'test', 'val']
data = []

# Loop through Image Directories to find images
for dir in image_dirs:
    dir_path = os.path.join(data_filepath, dir)
    # Loop through NORMAL and PNEUMONIA directories, and label images accordingly
    for label_name in ["NORMAL", "PNEUMONIA"]:
        label_dir = os.path.join(dir_path, label_name)
        for file_name in os.listdir(label_dir):
            file_path = os.path.join(label_dir, file_name)
            # Ensure it's an image file (basic check)
            if file_name.endswith(('.png', '.jpg', '.jpeg')):
                data.append([dir, file_path, label_name])

    
# Create DataFrame
df = pd.DataFrame(data, columns=["directory", "image_path", "label"])
df

,directory,image_path,label
0,train,./data/chest_xray\train\NORMAL\IM-0115-0001.jpeg,NORMAL
1,train,./data/chest_xray\train\NORMAL\IM-0117-0001.jpeg,NORMAL
2,train,./data/chest_xray\train\NORMAL\IM-0119-0001.jpeg,NORMAL
3,train,./data/chest_xray\train\NORMAL\IM-0122-0001.jpeg,NORMAL
4,train,./data/chest_xray\train\NORMAL\IM-0125-0001.jpeg,NORMAL
...,...,...,...
5851,val,./data/chest_xray\val\PNEUMONIA\person1949_bac...,PNEUMONIA
5852,val,./data/chest_xray\val\PNEUMONIA\person1950_bac...,PNEUMONIA
5853,val,./data/chest_xray\val\PNEUMONIA\person1951_bac...,PNEUMONIA
5854,val,./data/chest_xray\val\PNEUMONIA\person1952_bac...,PNEUMONIA


In [4]:

def resize_image(img_path, output_dir, target_size, img_name):
    '''Function to resize the supplied image.  
        img_path = path to the image for resizing
        output_dir = file path for saving the resized image
        target_size = target size of the image (i.e. (256,256))
        image_name = Name for the resized image'''
    img = Image.open(img_path)
    img_resized = img.resize(target_size, Image.LANCZOS)
    output_path = output_dir / img_name
    img_resized.save(output_path)
    

In [6]:
# Variables for use in the resize image function
sample_size = 1000
normal_output_dir = Path("./data/chest_xray/resized/normal")
pneumonia_output_dir = Path("./data/chest_xray/resized/pneumonia")
target_size = (128, 128)

# Spit out for Normal and P
normal_sample = df[df['label'] == 'NORMAL']['image_path'].sample(n=sample_size)
pneumonia_sample = df[df['label'] == 'PNEUMONIA']['image_path'].sample(n=sample_size)

# Create output directory if it doesn't exist
normal_output_dir.mkdir(parents=True, exist_ok=True)
pneumonia_output_dir.mkdir(parents=True, exist_ok=True)


# Loop through normal sample DF and process images.
i=0
for img_path in normal_sample:
    resize_image(Path(img_path), normal_output_dir, target_size, f'img_{i}.jpeg')
    i = i + 1


# Loop through normal sample DF and process images.
i=0
for img_path in pneumonia_sample:
    resize_image(Path(img_path), pneumonia_output_dir, target_size, f'img_{i}.jpeg')
    i = i + 1

# Setup Environment
This section of the Notebook sets up the StyleGAN model, fine-tunes it for use with X-Ray images, snapshots the model, and then uses the model for generating new images.   This portion of the notebook must be run with a GPU, and is recommended to run in Google Collab.

In [2]:
# Load Libraries used in this notebook
import torch
import os


In [3]:
# Verify GPU is avialable
print(torch.cuda.is_available())
print(f"Number of GPUs: {torch.cuda.device_count()}")

True
Number of GPUs: 1


In [4]:
# Install additional environment libraries used by StyleGan
!pip install timm==0.4.12 ftfy==6.1.1 ninja==1.10.2 opensimplex

In [5]:
# Clone the Style3GAN repository and verify it exists
repo_url = "https://github.com/NVlabs/stylegan3.git"
repo_dir = "../content/stylegan3"

# Remove the directory if it already exists to avoid conflicts
if os.path.exists(repo_dir):
    !rm -rf {repo_dir}

# Clone the repo
!git clone {repo_url} {repo_dir}

# Change directory to the repo
%cd {repo_dir}

# Verify the contents
!ls

Cloning into '../content/stylegan3'...
remote: Enumerating objects: 212, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 212 (delta 99), reused 90 (delta 90), pack-reused 49 (from 1)
Receiving objects: 100% (212/212), 4.16 MiB | 17.90 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/stylegan3
avg_spectra.py	 dnnlib      environment.yml  gui_utils    metrics	training       viz
calc_metrics.py  Dockerfile  gen_images.py    legacy.py    README.md	train.py
dataset_tool.py  docs	     gen_video.py     LICENSE.txt  torch_utils	visualizer.py


In [153]:
# Prepare Image data using StyleGan dataset tool
#   NOTE:  IMAGES YOU ARE USING TO FINE TUNE THE GAN MUST BE IN THE SOURCE
CMD = "python /content/stylegan3/dataset_tool.py "\
  "--source /content/data/normal "\
  "--dest /content/data/dataset/normal"

!{CMD}

 78% 731/938 [00:00<00:00, 1047.08it/s]Error: Image 00000/img00000760.png attributes must be equal across all images of the dataset.  Got:
  dataset width/cur image width: 128/128
  dataset height/cur image height: 128/128
  dataset channels/cur image channels: 1/3
 81% 760/938 [00:00<00:00, 1038.35it/s]


In [152]:
# Command for clearing out newly created dataset if things go wrong
#!rm -R /content/data/dataset/pneumonia*

# Fine-Tune Model

This section will begin the fine-tuning of the StyleGan.  This process can be quite lengthy depending on the environment and avialable infrastructure.
This notebook was run on a Google Collab "A100 GPU" runtime.  It was manually stopped once generated images looked indiscernible from real images.

In [154]:
# Modify these to suit your needs
EXPERIMENTS = "/content/data/experiments/pneumonia"
DATA = "/content/data/dataset/pneumonia"
SNAP = 10

# Build the command and run it
cmd = f"/usr/bin/python3 /content/stylegan3/train.py "\
  f"--snap {SNAP} --outdir {EXPERIMENTS} --data {DATA} --cfg stylegan2 --gpus 1 --batch 32 --gamma 6.6 "
!{cmd}


Training options:
{
  "G_kwargs": {
    "class_name": "training.networks_stylegan2.Generator",
    "z_dim": 512,
    "w_dim": 512,
    "mapping_kwargs": {
      "num_layers": 8
    },
    "channel_base": 32768,
    "channel_max": 512,
    "fused_modconv_default": "inference_only"
  },
  "D_kwargs": {
    "class_name": "training.networks_stylegan2.Discriminator",
    "block_kwargs": {
      "freeze_layers": 0
    },
    "mapping_kwargs": {},
    "epilogue_kwargs": {
      "mbstd_group_size": 4
    },
    "channel_base": 32768,
    "channel_max": 512
  },
  "G_opt_kwargs": {
    "class_name": "torch.optim.Adam",
    "betas": [
      0,
      0.99
    ],
    "eps": 1e-08,
    "lr": 0.002
  },
  "D_opt_kwargs": {
    "class_name": "torch.optim.Adam",
    "betas": [
      0,
      0.99
    ],
    "eps": 1e-08,
    "lr": 0.002
  },
  "loss_kwargs": {
    "class_name": "training.loss.StyleGAN2Loss",
    "r1_gamma": 6.6,
    "style_mixing_prob": 0.9,
    "pl_weight": 2,
    "pl_no_weight_grad

# Load and use Model

Models are snapshot every 10,000 generated images (10 ticks).  The following section loads the model from a .pkl file and generates x-ray images for use in other machine learning applications.

In [6]:
# Import additional libraries used for generating images with the StyleGAN 
import pickle
import numpy as np
import dnnlib
import torch_utils
from tqdm import tqdm
from PIL import Image


The images being generated are grayscale, which seems to be problematic for StyleGAN3.  To get around this, the following edit needs to be made to the gen_images.py file from the downlaoded StyelGan repo code.

**COMMENT OUT THESE TWO LINES**  
  #img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)  
  #PIL.Image.fromarray(img[0].cpu().numpy(), 'RGB').save(f'{outdir}/seed{seed:04d}.png')  


**PUT IN THESE TWO LINES**  
img = (img.permute(0, 1, 2, 3) * 127.5 + 128).clamp(0, 255).to(torch.uint8)  
PIL.Image.fromarray(img[0].cpu().numpy()[0]).save(f'{outdir}/seed{seed:04d}.png')  

In [20]:
# Path to the model
model_path = "/content/models/stylegan-penumonia-000240.pkl"

img_output_path = "/content/drive/MyDrive/GAN-Images/Pneumonia"

# Generate an images
cmd = f"/usr/bin/python3 /content/stylegan3/gen_images.py "\
  f"--outdir={img_output_path} --trunc=1 --seeds=1-1000 --network={model_path} "
!{cmd}


Loading networks from "/content/models/stylegan-penumonia-000240.pkl"...
Generating image for seed 1 (0/1000) ...
Setting up PyTorch plugin "bias_act_plugin"... /usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Done.
Setting up PyTorch plugin "upfirdn2d_plugin"... /usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Done.
Generating image for seed 2 (1/1000) ...
Generating image for seed 3 (2/1000) ...
Generating image for seed 4 (3/1000) ...
Generating image for seed 5 (4/1000) ...
Generating image for seed 6 (5/1000) ...
Generating image for seed 7 (6/1000) .